# Tokenization & Text Preprocessing

Everything BM25 and TF-IDF score happens *after* text has been cleaned and split into tokens. The preprocessing pipeline decides **what counts as a term** — and that choice quietly determines your retrieval quality. This notebook walks the pipeline end to end on a small corpus and measures the effect of each step.

## Index

1. [Why preprocessing matters](#s1)
2. [The raw data](#s2)
3. [Tokenization](#s3)
4. [Lowercasing & punctuation removal](#s4)
5. [Stopword removal — why dropping "the" and "is" helps BM25](#s5)
6. [Stemming vs lemmatization — "running" → "run" and the recall payoff](#s6)
7. [The full pipeline](#s7)
8. [Measuring the impact on retrieval](#s8)
9. [Notes for Arabic & morphologically rich languages](#s9)


<a id="s1"></a>
## 1. Why preprocessing matters

A ranker never sees raw text — it sees a **bag of tokens**. Two strings that mean the same thing (`"Running"`, `"runs"`, `"ran"`) become *different* tokens unless preprocessing collapses them. Every step below changes the vocabulary the index is built from:

| Step | What it does | Why it matters for ranking |
|------|--------------|----------------------------|
| Lowercasing | `"The"` → `"the"` | `"The"` and `"the"` stop being two separate terms |
| Punctuation removal | `"don't!"` → `don`, `t` | strips noise that fragments terms |
| Stopword removal | drop `the`, `is`, `a` | removes near-zero-signal terms that inflate document length |
| Stemming / lemmatization | `running` → `run` | lets a query for `run` match `running` (recall) |

Get this wrong and even a perfect scoring function returns bad results — the relevant document was tokenized into terms the query can never match.

In [1]:
import re
import numpy as np
import pandas as pd
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords

# one-time data downloads (no-ops if already present)
for pkg in ["wordnet", "omw-1.4", "stopwords"]:
    nltk.download(pkg, quiet=True)

# the same tokenizer used in the TF-IDF and BM25 notebooks
TOKEN = r"\w+(?:\.\w+)*"
def tokenize(text):
    return re.findall(TOKEN, text.lower())

<a id="s2"></a>
## 2. The raw data

Eight short, deliberately *messy* documents in `data/corpus.txt`: mixed case, punctuation, numbers, contractions, and — crucially — many **morphological variants** of the same root (`runs`/`ran`/`running`/`runner`, `studies`/`studied`/`student`, `investing`/`invested`/`investment`). Those variants are what stemming and lemmatization will collapse later.

In [2]:
with open("data/corpus.txt") as f:
    corpus = [line.strip() for line in f if line.strip()]

for i, doc in enumerate(corpus):
    print(f"[{i}] {doc}")

[0] The Quick Brown Fox runs every morning; it ran 5 miles yesterday, and it's STILL running strong!
[1] Marathon runners train for months. A good runner runs daily and rarely stops running.
[2] She studies Machine Learning at MIT. The student studied hard, and her studies are finally paying off!
[3] Investors are investing in tech stocks. The investor invested wisely, and investment returns grew 12%.
[4] Children were playing in the park; a child plays happily while the other kids played until dusk.
[5] Cats were sleeping on the warm mat. The cat slept all day and is still fast asleep.
[6] The U.S. economy grew 3.2% in 2023, according to a report released on Jan. 5th.
[7] E-mails, e-mails everywhere! Don't forget to check your e-mail — it's important, isn't it?


<a id="s3"></a>
## 3. Tokenization

Tokenization splits a string into terms. The regex `\w+(?:\.\w+)*` keeps runs of word-characters and lets internal dots survive (so `u.s.a` stays one token). Everything else — spaces, punctuation — becomes a separator. Because the pattern also lowercases inside `tokenize`, steps 3 and 4 happen together here, but we pull them apart below to see each effect.

In [3]:
sample = corpus[0]
print("raw:   ", sample)
print("tokens:", tokenize(sample))

raw:    The Quick Brown Fox runs every morning; it ran 5 miles yesterday, and it's STILL running strong!
tokens: ['the', 'quick', 'brown', 'fox', 'runs', 'every', 'morning', 'it', 'ran', '5', 'miles', 'yesterday', 'and', 'it', 's', 'still', 'running', 'strong']


<a id="s4"></a>
## 4. Lowercasing & punctuation removal

- **Lowercasing** makes matching case-insensitive: a query for `fox` should match `Fox` and `FOX`. Without it, those are three distinct terms and two of them never match the query.
- **Punctuation removal** stops punctuation from fragmenting or duplicating terms: `"morning;"`, `"strong!"`, `"it's"` should not become tokens like `morning;` or `strong!`.

The regex tokenizer already does both (it lowercases, and only captures word-characters). The cell below shows what we'd get *without* this normalization, to make the difference explicit.

In [4]:
raw_split = corpus[0].split()                         # naive whitespace split, no normalization
clean = tokenize(corpus[0])                            # lowercased + punctuation stripped

print("naive whitespace split:", raw_split)
print()
print("normalized tokens:     ", clean)
print()
# the concrete damage avoided:
print("'STILL' vs 'still'   ->", "STILL".lower(), "(case folded so it matches a 'still' query)")
print("'morning;' -> token   ->", tokenize("morning;"))
print("\"it's\"   -> tokens   ->", tokenize("it's"))

naive whitespace split: ['The', 'Quick', 'Brown', 'Fox', 'runs', 'every', 'morning;', 'it', 'ran', '5', 'miles', 'yesterday,', 'and', "it's", 'STILL', 'running', 'strong!']

normalized tokens:      ['the', 'quick', 'brown', 'fox', 'runs', 'every', 'morning', 'it', 'ran', '5', 'miles', 'yesterday', 'and', 'it', 's', 'still', 'running', 'strong']

'STILL' vs 'still'   -> still (case folded so it matches a 'still' query)
'morning;' -> token   -> ['morning']
"it's"   -> tokens   -> ['it', 's']


<a id="s5"></a>
## 5. Stopword removal — why dropping "the" and "is" helps BM25

**Stopwords** are ultra-common function words (`the`, `is`, `a`, `and`, `of`). They appear in almost every document, so they carry almost no information about *which* document is relevant. Removing them helps BM25 in three concrete ways:

1. **They have near-zero IDF.** BM25's IDF is $\log\!\left(1 + \frac{N - n + 0.5}{n + 0.5}\right)$. A term in nearly every document has $n \approx N$, driving IDF toward its floor — so stopwords contribute little signal but still add noise.
2. **They inflate document length.** BM25's length-normalization term $b\cdot\frac{|d|}{\text{avgdl}}$ depends on $|d|$. Padding documents with stopwords lengthens them, distorting the normalization for the terms that actually matter.
3. **They cost time and space.** Fewer tokens means a smaller index and faster scoring.

The cell below shows the IDF gap and the length reduction on our corpus.

In [5]:
STOPWORDS = set(stopwords.words("english"))
doc_tokens = [tokenize(d) for d in corpus]

def idf(term, docs):
    N = len(docs)
    n = sum(1 for d in docs if term in d)
    return np.log(1 + (N - n + 0.5) / (n + 0.5))

idf_table = pd.DataFrame(
    [(w, "stopword" if w in STOPWORDS else "content", round(idf(w, doc_tokens), 3))
     for w in ["the", "and", "a", "is", "running", "investment", "marathon"]],
    columns=["term", "kind", "idf"],
).sort_values("idf")
print(idf_table.to_string(index=False))

without_sw = [[w for w in d if w not in STOPWORDS] for d in doc_tokens]
raw_n = sum(len(d) for d in doc_tokens)
new_n = sum(len(d) for d in without_sw)
print(f"\ntotal tokens: {raw_n}  ->  {new_n} after stopword removal  ({100*(raw_n-new_n)/raw_n:.0f}% smaller)")
print("example doc[0] without stopwords:", without_sw[0])

      term     kind   idf
       the stopword 0.325
       and stopword 0.492
         a stopword 0.944
   running  content 1.281
        is stopword 1.792
investment  content 1.792
  marathon  content 1.792

total tokens: 132  ->  84 after stopword removal  (36% smaller)
example doc[0] without stopwords: ['quick', 'brown', 'fox', 'runs', 'every', 'morning', 'ran', '5', 'miles', 'yesterday', 'still', 'running', 'strong']


> **Honest caveat on a tiny corpus.** With only 8 documents, the IDF signal is noisy — `the` and `and` clearly bottom out, but a word like `is` happens to appear in few of *these* documents, so its IDF looks high here. In a real corpus of thousands of documents, every stopword converges to the low-IDF floor. The *mechanism* is what matters: words that appear everywhere can't discriminate.

<a id="s6"></a>
## 6. Stemming vs lemmatization — "running" → "run" and the recall payoff

A query for `run` should find documents about `running`, `runs`, and `ran`. But those are four different tokens — so without normalization the query matches **none** of them. Two techniques collapse word forms to a common base:

- **Stemming** chops off suffixes with crude rules (Porter's algorithm). Fast, language-specific rules, no dictionary. Downsides: it **over-stems** (`organization` → `organ`) and produces **non-words** (`studies` → `studi`), and it misses irregular forms (`ran` stays `ran`).
- **Lemmatization** maps a word to its dictionary **lemma** using a vocabulary and part-of-speech. Returns real words, handles irregulars (`ran` → `run`, `better` → `good`) — but **needs the correct POS** and a language resource, and is slower.

The table below contrasts both on tricky words. Watch the `ran` and `better` rows (where stemming fails) and the `studi`/`organ` rows (where stemming over-reaches).

In [6]:
ps = PorterStemmer()
wl = WordNetLemmatizer()

words = ["running", "runs", "ran", "runner", "studies", "studied", "better", "universities", "organization"]
comp = pd.DataFrame(
    [(w, ps.stem(w), wl.lemmatize(w, "v"), wl.lemmatize(w, "n")) for w in words],
    columns=["word", "porter_stem", "lemma (verb)", "lemma (noun)"],
)
print(comp.to_string(index=False))

        word porter_stem lemma (verb) lemma (noun)
     running         run          run      running
        runs         run          run          run
         ran         ran          run          ran
      runner      runner       runner       runner
     studies       studi        study        study
     studied       studi        study      studied
      better      better       better       better
universities     univers universities   university
organization       organ organization organization


Notice the two failure modes side by side:
- **`ran`**: Porter leaves it as `ran` (it only strips suffixes); the verb lemma correctly gives `run`. Lemmatization wins on irregulars.
- **`organization` / `universities`**: Porter mangles them to `organ` / `univers`; the lemma keeps the real word. Stemming can over-merge unrelated words (hurting *precision*), while still helping *recall*.
- **POS matters**: `running` lemmatizes to `run` as a *verb* but stays `running` as a *noun*. Lemmatization without the right POS silently does nothing.

<a id="s7"></a>
## 7. The full pipeline

Putting the steps in order: **tokenize → remove stopwords → stem (or lemmatize)**. Order matters — remove stopwords *before* stemming, since stemming could otherwise turn a content word into something that looks like a stopword (or vice versa).

In [7]:
def preprocess(text, use_stemming=True, remove_stopwords=True):
    tokens = tokenize(text)                                  # 3 + 4: tokenize, lowercase, strip punctuation
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS]   # 5
    if use_stemming:
        tokens = [ps.stem(t) for t in tokens]                # 6 (stemming branch)
    return tokens

print("raw:       ", corpus[1])
print("preprocessed:", preprocess(corpus[1]))

raw:        Marathon runners train for months. A good runner runs daily and rarely stops running.
preprocessed: ['marathon', 'runner', 'train', 'month', 'good', 'runner', 'run', 'daili', 'rare', 'stop', 'run']


<a id="s8"></a>
## 8. Measuring the impact on retrieval

The point of all this is **recall**: how many relevant documents a query can even reach. We label a handful of single-word queries with their relevant documents (`data/queries.txt`) and measure recall under three regimes — raw tokens, stemmed, lemmatized — using simple term-membership matching.

Because every query word appears only in an *inflected* form in the corpus (`run` never appears literally — only `runs`/`ran`/`running`), raw matching should score near zero, and normalization should rescue it.

In [8]:
queries = []
with open("data/queries.txt") as f:
    for line in f:
        term, rel = line.strip().split("\t")
        queries.append((term, set(int(x) for x in rel.split(","))))

def mean_recall(normalize):
    """normalize: function token -> base form (identity for raw)."""
    norm_docs = [[normalize(w) for w in d] for d in doc_tokens]
    recalls = []
    for term, relevant in queries:
        q = normalize(term)
        found = {i for i, d in enumerate(norm_docs) if q in d}
        recalls.append(len(found & relevant) / len(relevant))
    return np.mean(recalls)

identity = lambda w: w
stem = lambda w: ps.stem(w)
lemma = lambda w: wl.lemmatize(w, "v")

summary = pd.DataFrame([
    {"regime": "raw tokens",   "mean recall": round(mean_recall(identity), 3)},
    {"regime": "stemmed",      "mean recall": round(mean_recall(stem), 3)},
    {"regime": "lemmatized",   "mean recall": round(mean_recall(lemma), 3)},
]).set_index("regime")
print("queries:", [q for q, _ in queries])
summary

queries: ['run', 'study', 'invest', 'play', 'sleep']


,mean recall
regime,
raw tokens,0.0
stemmed,1.0
lemmatized,1.0


The jump from **raw → stemmed/lemmatized** is the whole argument for this preprocessing step: without it, a user searching `run` would never see the document about `running` and `ran`. This is also the recall/precision trade-off in miniature — aggressive stemming maximizes recall but can merge unrelated words and cost precision; lemmatization is gentler and keeps real words at the price of needing POS and a dictionary.

<a id="s9"></a>
## 9. Notes for Arabic & morphologically rich languages

Everything above is tuned for English. Arabic (and other morphologically rich languages) breaks several assumptions, which is why off-the-shelf English pipelines retrieve poorly on Arabic text:

- **No case, but orthographic normalization is essential.** There is no lowercasing, but you *must* normalize variants that users type interchangeably: alef forms (`أ`, `إ`, `آ` → `ا`), taa marbuta (`ة` ↔ `ه`), alef maqsura (`ى` ↔ `ي`), and you must strip **diacritics (tashkeel)** and the elongation character **tatweel** (`ـ`). Skipping this fragments the same word into many tokens.
- **Whitespace tokenization is not enough.** Arabic attaches **clitics** directly to words — conjunctions (`و`), the definite article (`ال`), prepositions (`ب`, `ل`, `ك`), and pronoun suffixes. The single written form `وبالكتاب` ("and with the book") is one whitespace token but several morphemes. Proper tokenization needs **morphological segmentation** (e.g., Farasa, CAMeL Tools).
- **Stopwords are often clitics, not standalone words.** Removing a stopword list of separate words misses function morphemes glued onto content words — segmentation has to happen first.
- **Stemming vs lemmatization is a sharper trade-off.** Arabic is **templatic** (root + pattern). *Light stemmers* strip affixes; *root extractors* (e.g., ISRI) reduce `كتب`, `كاتب`, `مكتوب`, `كتاب` to the root `كتب`, which boosts recall enormously but can over-merge semantically distinct words (hurting precision). True **lemmatization** needs a morphological analyzer (Farasa, MADAMIRA, CAMeL Tools) and usually outperforms root-stemming for retrieval quality.

The general lesson holds in every language: **the preprocessing pipeline defines your vocabulary, and your vocabulary defines what your ranker can ever match.** The specific operations just have to match the morphology of the language.